## Langchain v1.0을 이용해서 챗봇 구현하기

`create_agent()`를 활용해 에이전트를 구현해 보겠습니다.

In [1]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver


import os
from dotenv import load_dotenv

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(dotenv_path="../.env")

# 모델 이름
model_name = os.getenv("model")

# LLM 정의
model = ChatOpenAI(
    model=model_name,
    api_key=os.getenv("credential_key"),
    temperature=0.7,
)

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


#### 1. 도구 활용하기

그냥 모델은 도구를 사용할 수 없기 때문에 아래와 같은 질문들에 제대로 대답할 수 없습니다.

In [3]:
response = model.invoke("오늘 서울 날씨는 어때?")
print(response.content)

죄송하지만 저는 **실시간 날씨 정보**를 바로 조회할 수는 없어요.  
대신 아래 방법으로 **오늘 서울 날씨**를 빠르게 확인할 수 있습니다:

- **네이버/구글에 “서울 날씨”** 검색
- **기상청 날씨누리** 확인
- 스마트폰 기본 날씨 앱 확인

원하시면 제가 대신  
1) **서울 날씨 확인 방법**을 바로 안내해드리거나,  
2) **오늘 외출 복장 추천**을 날씨에 맞춰 드릴 수 있어요.


그렇기 때문에 2가지가 필요합니다.

1. 모델이 사용할 수 있는 도구(Tool)
2. 모델이 도구를 사용할 수 있도록 에이전트로 변환.

#### Tool

Tool이란, LLM이 사용할 수 있는 도구를 말합니다. @Tool 데코레이션을 사용해 구현할 수 있습니다.

LLM에게 어떤 경우에 어떻게 사용할 수 있는지 자세히 설명해주는 것이 좋습니다.

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str: # 문자열(str)을 받아서 문자열을 반환한다는 의미.
    """특정 장소의 날씨 정보를 알려주는 기능""" # LLM이 이 설명을 읽고 도구를 사용합니다.
    return f"It's sunny in {location}"

`create_agent()`를 통해 LLM을 agent로 변환해 줄 수 있습니다!

In [5]:
# agent로 변환 : 이제 llm이 도구를 스스로 쓸 수 있게 됩니다.
agent = create_agent(
    model,
    tools=[get_weather],
    system_prompt="너는 기상캐스터로써, 사용자가 물어보는 지역에 대한 날씨 정보를 알려줘야 한다."
)

다시 한 번 날씨를 물어볼까요?

In [21]:
messages = [
    {"role": "user", "content": "오늘 서울의 날씨는 어떤가요?"},
]

response = agent.invoke(
    {"messages": messages}
)

print("모델의 tool calling :", response['messages'][1].tool_calls)
print("도구 사용 결과 : ", response['messages'][2])
print('\n')
print(response['messages'][3].content)

모델의 tool calling : [{'name': 'get_weather', 'args': {'location': '서울'}, 'id': 'call_HqHoNUxUsX98PDvRwdXwP4MC', 'type': 'tool_call'}]
도구 사용 결과 :  content="It's sunny in 서울" name='get_weather' id='06348c44-60ba-4715-b7ac-cd261c577c9c' tool_call_id='call_HqHoNUxUsX98PDvRwdXwP4MC'


오늘 서울은 **맑은 날씨**입니다.  
대체로 **햇볕이 좋은 편**이에요.

원하시면 제가 **기온, 강수확률, 바람**까지 같이 알려드릴게요.


모델이 `get_weather()` 함수를 스스로 호출하고 결과를 확인한 뒤에 '서울의 날씨는 **맑음**' 이라고 대답한 것을 확인할 수 있습니다!

이렇게 도구를 정의하고, 설명만 LLM이 이해할 수 있도록 작성을 해둔다면 LLM이 스스로 도구를 사용합니다!

실습1 : 계산기를 사용하는 LLM

예로부터 LLM이 기초 연산에 약점을 보이는 문제가 많았었는데요,

직접 사칙연산을 수행하는 함수를 구현하고 LLM이 tool로 활용해 연산을 수행할 수 있도록 해봅시다!

`calculator(a: int, b: int, operator: str)->float:` : 2개의 변수 a, b를 입력 받아, operator 연산을 수행한다.

In [ ]:
# calculator 함수 완성
@tool
def calculator(a: int, b: int, operator: str) -> float:
    return

agent = create_agent(
    model,
    tools=[calculator],
    system_prompt="너는 친절한 계산기야. 사용자가 질문한 사칙연산을 빠르고 정확하게 답변해야 해."
)

questions = [
    [{"role": "user", "content": "3178 + 254의 계산결과는?"},],
    [{"role": "user", "content": "5137 - 33은?"},],
    [{"role": "user", "content": "223 x 317이 뭐야?"},],
    [{"role": "user", "content": "57을 8로 나누면 뭐야?"},],
]

for question in questions:
    response = agent.invoke(
        {"messages": question}
    )
    print(response['messages'][-1].content)

3178 + 254 = **3432**
5104
223 × 317 = 70,691 입니다.
57 ÷ 8 = 7.125 입니다.


이번엔 도구를 여러개 쥐어주고, 에이전트가 정말 각 상황에 적절한 도구를 호출하는 지도 확인해 봅시다!

In [7]:
@tool
def get_weather(location: str) -> str: # 문자열(str)을 받아서 문자열을 반환한다는 의미.
    """특정 장소의 날씨 정보를 알려주는 함수""" # LLM이 이 설명을 읽고 도구를 사용합니다.
    return f"It's sunny in {location}"

@tool
def calculator(a: int, b: int, operator: str) -> float:
    """
    2개의 정수 a, b를 입력 받으면 연산자(operator) string에 
    해당하는 연산을 수행하고 그 결과를 반환하는 함수.
    operator에는 'plus', 'minus', 'multiply', 'divide' 4가지만 들어갈 수 있다.
    """
    if operator == "plus":
        return a + b
    if operator == "minus":
        return a - b
    if operator == "multiply":
        return a * b
    if operator == "divide":
        return a / b

@tool
def get_temperature(location: str) -> str:
    """특정 장소의 기온 정보를 알려주는 함수"""
    return f"{location}의 기온은 17도 입니다."

agent = create_agent(
    model,
    tools=[get_temperature, get_weather, calculator],
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "오늘 부산의 날씨와 기온을 알려줘. 기온 알려줄 때는 10도를 더해서 알려줘."}
    ]}
)

In [8]:
from langchain_core.messages import AIMessage

for message in response['messages']:
    if isinstance(message, AIMessage) and message.tool_calls:
        for tool_call in message.tool_calls:
            print("tool calling :", tool_call['name'])
    if message.content:
        print("content :", message.content)

content : 오늘 부산의 날씨와 기온을 알려줘. 기온 알려줄 때는 10도를 더해서 알려줘.
tool calling : get_weather
tool calling : get_temperature
content : It's sunny in 부산
content : 부산의 기온은 17도 입니다.
tool calling : calculator
content : 27
content : 오늘 부산의 날씨는 **맑음**입니다.  
기온은 **27도**입니다.


#### Agents.md

시스템 프롬프트란? : AI의 행동지침을 정의하는 프롬프트.

일반적으로 에이전트의 행동 방침은 코드와 분리하여 `agents.md`라는 파일에 작성을 하고 로드하는 식으로 구현합니다.

이는 아래와 같은 이점들 때문입니다.

1. 코드가 깔끔해진다 : 코드 안에 프롬프트가 길게 들어가면 가독성이 떨어짐.
2. 수정이 편하다 : 프롬프트 바꿀 때마다 코드를 수정할 필요 없이 `agents.md`만 수정하면 됨.
3. 협업/버전 관리가 쉬움 : 프롬프트 변경 이력을 확인하기 용이하고, 개발자가 아닌 사람도 쉽게 편집할 수 있음.

In [ ]:
agent = create_agent(
    model,
    tools=[get_weather, get_temperature, calculator],
    system_prompt=
    """
    ## 역할
    - 사용자의 질문에 정확하고 간결하게 답하는 AI 어시스턴트

    ## 규칙
    - 핵심 내용만 짧게 전달하고, 불필요한 부연 설명은 생략
    - 도구 실행 결과는 그대로 인용하며, 임의로 수정하지 않기
    - 계산이나 조회가 필요한 경우 반드시 도구를 사용하고 추측으로 답하지 않기
    - 불확실한 내용은 단정하지 않기
    """
)

# 이랬던 코드가...

In [9]:
# 이렇게 짧게 바뀜

with open("agents.md", encoding="utf-8") as f:
    system_prompt = f.read()

tools = [get_weather, get_temperature, calculator]

agent = create_agent(
    model,
    tools=tools,
    system_prompt=system_prompt
)

ChatPromptTemplate과 함께 활용하고 싶다면 아래와 같이 활용할 수 있습니다.

In [ ]:
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

system_prompt = open("agents.md").read()

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=system_prompt),        # 고정 (변수 치환 없음)
    HumanMessagePromptTemplate.from_template("{input}")  # 변수는 여기만
])

#### 메모리 관리

앞서도 살펴봤듯이, 기본적으로는 이전의 대화 내역을 모두 모델에게 입력하는 식으로 모델의 메모리를 유지합니다.

In [44]:
messages = [
    {"role": "user", "content": "지금부터 네 이름은 트럼프야."},
    {"role": "assistant", "content": "알겠습니다! 지금부터는 저를 트럼프라고 불러주세요! 무엇을 도와드릴까요?"},
    {"role": "user", "content": "트럼프, 나 오늘 주식을 다 잃었어. 어떡하지?"}
]

response = agent.invoke(
    {"messages": messages}
)

print(response['messages'][-1].content)

많이 힘드셨겠어요. 지금은 **추가 매매를 멈추고** 먼저 상황을 정리하는 게 중요합니다.

- **오늘은 거래 앱/차트를 닫기**
- **손실 금액을 정확히 확인**
- **현금 유동성부터 확보**
- **빚으로 투자한 게 있으면 상환 계획 확인**
- **다음엔 감당 가능한 금액만 투자**하기

감정이 너무 크게 흔들리면 혼자 버티지 말고 **가까운 사람에게 바로 말하세요**.  
만약 지금 **스스로를 해칠 생각**이 조금이라도 든다면, **즉시 112/119 또는 자살예방상담전화 109**에 연락하세요.

원하시면 제가 바로  
1) **손실 정리 체크리스트**  
2) **내일 할 일 3가지**  
로 짧게 정리해드릴게요.


하지만 귀찮죠?

직접 대화를 전부 `messages` 리스트에 기록하고, 다시 꺼내오고...

지금은 사용자가 1명이니 충분히 관리할 수 있다곤 해도, 여러 명의 사용자가 생기면 각 사용자마다 대화를 따로 따로 관리해야 하고 이는 굉장히 번거로운 일이 될 겁니다!

당연히 LangChain에서는 이를 편리하게 할 수 있는 메모리 기능을 따로 제공하고 있습니다. 바로 `InMemorySaver`죠!

In [10]:
memory = InMemorySaver()

agent = create_agent(
    model,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=memory # checkpointer의 인자로 전달하여 사용.
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕! 내 이름은 민수야. 잘 부탁해!"}]},
    {"configurable": {"thread_id": "1"}} # 사용자 관리를 위한 thread_id를 추가 정보로 제공한다.
)

print(response['messages'][-1].content)

안녕하세요, 민수님! 반가워요. 잘 부탁드립니다.


In [11]:
# 메모리 내용을 출력해 보면 대화 내용들이 모두 `memory` 변수에 잘 저장되어 있는 것을 확인할 수 있습니다.
for message in memory.get(config={"configurable": {"thread_id": "1"}})['channel_values']['messages']:
    print(message.content)

안녕! 내 이름은 민수야. 잘 부탁해!
안녕하세요, 민수님! 반가워요. 잘 부탁드립니다.


In [55]:
# 이제 개발자가 직접 이전 대화 기록들을 입력해 주지 않아도 이전 대화 내용을 기억합니다.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

민수님이세요.


`thread_id`를 이용해 여러 대화를 각각 관리하는 것도 훨씬 용이합니다!

In [56]:
# thread_id=1 : 민수 / thread_id=2 : 철수

response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 철수야. 오늘 날씨 어때?"}]},
    {"configurable": {"thread_id": "2"}}
)

print(response['messages'][-1].content)

오늘 날씨는 맑아요.


In [57]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    {"configurable": {"thread_id": "2"}}
)

print(response['messages'][-1].content)

철수예요.


`InMemorySaver()`는 단기 메모리입니다. 따라서 프로그램이 종료되면 사라지게 되죠.

이걸 프로그램이 종료된 뒤에도 기억하게 하기 위해선 해당 정보들을 따로 저장해야 합니다.

실제 서비스에서는 LangGraph의 Store와 DB를 연동해서 사용하지만 지금은 해당 정보들을 기록할 json 파일을 만들어 보도록 하겠습니다.

사용자에 대한 정보를 기록할 수 있는 txt 파일을 만든 뒤, tool을 이용해서 해당 파일을 관리할 겁니다!

In [15]:
from langchain_core.runnables import RunnableConfig

@tool
def save_user_info(info: str) -> str:
    """사용자에 대한 정보를 기록합니다. 추후 대화에 도움이 될 만한 사용자 정보를 기록하는데 사용하세요."""
    
    with open(f"user_profile.txt", "w") as f:
        f.write(info)
    
    return "사용자 정보를 저장했습니다."

@tool
def load_user_info() -> str:
    """저장된 사용자 정보를 불러옵니다."""
    
    try:
        with open(f"user_profile.txt", "r") as f:
            return f.read()
    except FileNotFoundError:
        return "저장된 정보가 없습니다."

In [16]:
agent = create_agent(
    model,
    tools=[save_user_info, load_user_info],
    checkpointer=InMemorySaver()
)

In [17]:
response = agent.invoke(
    {
        "messages": {"role": "user", "content": "안녕, 난 민희야. 앞으로 잘 부탁해!"},
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

안녕 민희! 반가워 😊  
앞으로 잘 부탁해. 편하게 이것저것 물어봐줘!


`user_profile.txt` 파일을 확인해 봅시다!